In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
(x_train, _), (_, _) = mnist.load_data()

x_train = x_train.reshape(-1, 28, 28, 1).astype('float32')
x_train = (x_train - 127.5) / 127.5

print("Training data shape:", x_train.shape)

Training data shape: (60000, 28, 28, 1)


In [3]:
latent_dim = 100

generator = keras.Sequential([
    layers.Dense(7 * 7 * 128, input_shape=(latent_dim,)),
    layers.LeakyReLU(),
    layers.Reshape((7, 7, 128)),
    
    layers.Conv2DTranspose(
        64, (5, 5),
        strides=(2, 2),
        padding='same'
    ),
    layers.LeakyReLU(),
    
    layers.Conv2DTranspose(
        1, (5, 5),
        strides=(2, 2),
        padding='same',
        activation='tanh'
    )
])

generator.summary()

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 6272)                │         633,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu (LeakyReLU)              │ (None, 6272)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape (Reshape)                    │ (None, 7, 7, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_transpose (Conv2DTranspose)   │ (None, 14, 14, 64)          │         204,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_1 (LeakyReLU)            │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_transpose_1 (Conv2DTranspose) │ (None, 28, 28, 1)           │           1,601 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 839,937 (3.20 MB)

 Trainable params: 839,937 (3.20 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
discriminator = keras.Sequential([
    layers.Conv2D(
        64, (5, 5),
        strides=(2, 2),
        padding='same',
        input_shape=(28, 28, 1)
    ),
    layers.LeakyReLU(),
    layers.Dropout(0.3),
    
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid')
])

discriminator.summary()

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 14, 14, 64)          │           1,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_2 (LeakyReLU)            │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 12544)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │          12,545 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,209 (55.50 KB)

 Trainable params: 14,209 (55.50 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
loss_function = keras.losses.BinaryCrossentropy()

generator_optimizer = keras.optimizers.Adam(0.0002)
discriminator_optimizer = keras.optimizers.Adam(0.0002)

In [6]:
@tf.function
def train_step(images):
    
    noise = tf.random.normal([128, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        fake_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(fake_images, training=True)

        gen_loss = loss_function(
            tf.ones_like(fake_output),
            fake_output
        )

        real_loss = loss_function(
            tf.ones_like(real_output),
            real_output
        )

        fake_loss = loss_function(
            tf.zeros_like(fake_output),
            fake_output
        )

        disc_loss = real_loss + fake_loss

    gen_gradients = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    disc_gradients = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(gen_gradients, generator.trainable_variables)
    )

    discriminator_optimizer.apply_gradients(
        zip(disc_gradients, discriminator.trainable_variables)
    )

    return gen_loss, disc_loss

In [7]:
batch_size = 128

dataset = tf.data.Dataset.from_tensor_slices(
    x_train
).shuffle(60000).batch(batch_size)

print("Dataset ready!")

Dataset ready!


In [ ]:
epochs = 10

for epoch in range(epochs):

    gen_loss = 0
    disc_loss = 0

    for images in dataset:
        gen_loss, disc_loss = train_step(images)

    print(
        f"Epoch {epoch + 1}/{epochs} - "
        f"Generator Loss: {gen_loss:.4f} - "
        f"Discriminator Loss: {disc_loss:.4f}"
    )

Epoch 1/10 - Generator Loss: 1.3536 - Discriminator Loss: 0.6250
Epoch 2/10 - Generator Loss: 1.2143 - Discriminator Loss: 0.7563
Epoch 3/10 - Generator Loss: 0.7416 - Discriminator Loss: 1.4176
Epoch 4/10 - Generator Loss: 0.8682 - Discriminator Loss: 1.5088
Epoch 5/10 - Generator Loss: 0.8251 - Discriminator Loss: 1.3245


In [ ]:
noise = tf.random.normal([16, latent_dim])

generated_images = generator(
    noise,
    training=False
)

generated_images = (generated_images + 1) / 2

plt.figure(figsize=(6, 6))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(
        generated_images[i, :, :, 0],
        cmap='gray'
    )
    plt.axis('off')

plt.suptitle("Generated Images by GAN")
plt.tight_layout()
plt.show()

In [ ]:
noise = tf.random.normal([16, latent_dim])

generated_images = generator(
    noise,
    training=False
)

generated_images = (generated_images + 1) / 2

plt.figure(figsize=(6, 6))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(
        generated_images[i, :, :, 0],
        cmap='gray'
    )
    plt.axis('off')

plt.suptitle("Generated Images by GAN")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(
        (x_train[i, :, :, 0] + 1) / 2,
        cmap='gray'
    )
    plt.axis('off')
    plt.subplot(2, 8, i + 9)
    plt.imshow(
        generated_images[i, :, :, 0],
        cmap='gray'
    )
    plt.axis('off')
plt.suptitle("Top: Real Images | Bottom: Generated Images")
plt.tight_layout()
plt.show()